# ⏬ Agentic — Example

- Creates a text file with content to be injected into prompt
- Creates a prompt template that references the text file content
- Creates and invokes the agent harness with the prompt and text file

---

### ▶️ Setup environment
> ❗ The following are user instructions

#### ➖ Run Installation script
```bash
./util/setup
```

#### ➖ (_optional_) Ensure environment vars
```bash
AGENT_INFERENCE_URL = 10.0.0.6:11434/v1
AGENT_INFERENCE_KEY = nosuch
AGENT_INFERENCE_LLM = huihui_ai/Qwen3.8-abliterated:27b`
AGENT_INFERENCE_VLM = huihui_ai/Qwen3.8-abliterated:27b`
```

#### ➖ (_optional_) Ensure configuration file
```yaml
# Interpolation to defer to environment vars
inference: # a group flattens onto inferenece_*
  url: ${oc.env:AGENT_INFERENCE_URL}
  key: ${oc.env:AGENT_INFERENCE_KEY}
  llm: ${oc.env:AGENT_INFERENCE_LLM}
  vlm: ${oc.env:AGENT_INFERENCE_VLM}
```

---

### ▶️ Create input content (_text file_)
> ❗ Input content can be raw text or a file path

In [ ]:
from pathlib import Path

INPUT_FILE_PATH = Path("EXAMPLE_INPUT.md")

INPUT_FILE_PATH.write_text("""
# Meeting notes
- The agent harness exposes a single `Agent` class.
- Sessions remember their conversation between runs.
- Every tool call is streamed with its arguments and outcome.
"""
)

---

### ▶️ Create and run agent (_new session_)

In [ ]:
from agent import Agent

# Create prompt that references the input content
PROMPT_TEMPLATE_1 = """
Summarise the following notes as bullet points:
{{ config.input }}
"""

# Create and run the harness against new session
agent_1 = Agent()
result_1 = await agent_1.run(PROMPT_TEMPLATE_1, input=INPUT_FILE_PATH)

# Display the results
print("done:", "ok" if result_1.ok else result_1.error)
print("error:", result_1.error)
print("output:", result_1.output)
print("session:", result_1.session_id)
print("tool calls:", [call.signature() for call in result_1.tools])


---

### ▶️ Create and run agent (_last session_)

In [ ]:
from agent import Agent

# Create prompt that references the input content
PROMPT_TEMPLATE_2 = """
How many bullet points did you produce, and what was the first one?
{{ config.input }}
"""

# Create and run the harness against last session
agent_2 = Agent()
result_2 = await agent_2.run(PROMPT_TEMPLATE_2, input=INPUT_FILE_PATH, session=result_1.session_id)

# Display the results
print("done:", "ok" if result_2.ok else result_2.error)
print("error:", result_2.error)
print("output:", result_2.output)
print("session:", result_2.session_id, "(same as before)" if result_2.session_id == result_1.session_id else "(new!)")
print("tool calls:", [call.signature() for call in result_2.tools])